# 🎬 पावर ओरिएंटर हब 2.0 – फ़ोटो से हिलता वीडियो (फ़्री)
**ये मशीन आपके अपने अकाउंट के फ़्री GPU पर चलती है।** आपकी फ़ोटो किसी और के पास नहीं जाती।

### 🆕 नया क्या है
- 💎 **बेहतर मॉडल:** Wan 2.2 **14B** Image-to-Video (Lightning, 4 स्टेप) – ज़्यादा असली मूवमेंट। *(Kaggle पर चलता है, Colab फ़्री में RAM कम है)*
- ⏱️ **लंबा वीडियो:** 5 / 10 / 15 / 20 सेकंड। हर लाइन में अलग सीन लिख सकते हो।
- 🇮🇳 **हिंदी में लिखो:** प्रॉम्प्ट अपने आप अंग्रेज़ी में बदलेगा + तैयार मूवमेंट बटन।
- 🔊 **आवाज़:** वीडियो देखकर AI ख़ुद आवाज़ बनाएगा (MMAudio), या अपना म्यूज़िक डालो।

### चालू करने का तरीक़ा
- **Kaggle में (बेहतर):** दाईं ओर `Settings → Accelerator → GPU T4 x2` और `Internet → On` करो, फिर ▶ दबाओ।
- **Colab में:** ऊपर `Runtime → Change runtime type → T4 GPU` चुनो, फिर नीचे वाले सेल का ▶ दबाओ।
- 8–15 मिनट बाद नीचे एक **`https://….gradio.live`** लिंक आएगा। उसे खोलो, वहीं वीडियो बनाओ।

### ⏱️ सच्ची जानकारी (T4 फ़्री GPU, हर 5 सेकंड वीडियो पर)
- ⚡ 5B: **480p में 4–6 मिनट**, 720p में 9–12 मिनट।
- 💎 14B: **480p में लगभग 12–18 मिनट**। 20 सेकंड का वीडियो = लगभग 1 घंटा।
- पहली बार मॉडल डाउनलोड: 5B ≈ 5 मिनट, 14B ≈ 15–25 मिनट (लगभग 40 GB)।
- फ़्री GPU की हफ़्ते की सीमा होती है (Kaggle में 30 घंटे)। काम पूरा हो जाए तो सेशन बंद कर दो।

### 🚫 नियम
असली लोगों के नक़ली वीडियो बनाना, किसी को बदनाम करना, अश्लील या हिंसक कंटेंट बनाना **मना है**। हर वीडियो पर "AI Generated" लिखा आता है। म्यूज़िक सिर्फ़ वही डालो जिसका हक़ आपके पास हो।

*Claude AI की मदद से बनाया गया। मॉडल: Wan 2.2 (Apache 2.0), MMAudio। इंजन: WanGP (deepbeepmeep)।*

In [ ]:
# ===== पावर ओरिएंटर हब 2.0 – एक बटन वाली मशीन =====
# नया: 💎 14B बेहतर मॉडल • ⏱️ 20 सेकंड तक लंबा वीडियो • 🇮🇳 हिंदी प्रॉम्प्ट • 🔊 AI आवाज़ / अपना म्यूज़िक
import os, sys, subprocess, shutil, glob, json, time, uuid, re
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')   # Kaggle के 2 GPU में से पहला
def sh(c): return subprocess.run(c, shell=True).returncode
def out(c): return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()

# ---- 1) कौन सी जगह: Colab या Kaggle ----
if os.path.exists('/kaggle'): BASE, PLACE = '/kaggle/temp', 'Kaggle'
elif os.path.exists('/content'): BASE, PLACE = '/content', 'Colab'
else: BASE, PLACE = os.path.expanduser('~/poh'), 'Computer'
os.makedirs(BASE, exist_ok=True)
print('📍 जगह:', PLACE)

# ---- 2) GPU, RAM, डिस्क जाँच ----
g = out('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
if not g:
    raise SystemExit('❌ GPU नहीं मिला!\n Colab: Runtime → Change runtime type → T4 GPU\n Kaggle: Settings → Accelerator → GPU T4 x2\n फिर दोबारा ▶ दबाओ।')
RAM_GB = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
DISK_GB = shutil.disk_usage(BASE).free / 1e9
print(f'✅ GPU: {g.splitlines()[0]} | RAM: {RAM_GB:.0f} GB | ख़ाली डिस्क: {DISK_GB:.0f} GB')
BIG_OK = RAM_GB >= 24 and DISK_GB >= 45      # 14B मॉडल के लिए कम से कम इतना चाहिए
if not BIG_OK:
    print('ℹ️ इस मशीन पर सिर्फ़ ⚡ तेज़ (5B) मॉडल चलेगा। 💎 14B के लिए Kaggle इस्तेमाल करो।')

# ---- 3) मशीन इंस्टॉल (पहली बार 5-10 मिनट) ----
WG = BASE + '/Wan2GP'
WG_COMMIT = '38758d3c73366422efcf609abb63cab8385f3552'   # जाँचा हुआ वर्ज़न, ताकि अचानक कुछ न टूटे
if not os.path.exists(WG + '/wgp.py'):
    print('⏳ मशीन डाउनलोड हो रही है...')
    sh(f'git clone -q --filter=blob:none https://github.com/deepbeepmeep/Wan2GP {WG}')
    sh(f'cd {WG} && git checkout -q {WG_COMMIT}')
os.chdir(WG)
if not os.path.exists('.poh2_ok'):
    print('⏳ ज़रूरी सामान इंस्टॉल हो रहा है (5-8 मिनट)...')
    sh("grep -viE '^(torch|torchvision|torchaudio)([=<> ]|$)' requirements.txt > req_poh.txt")
    if sh('pip install -q -r req_poh.txt') != 0:
        # कोई एक पैकेज फ़ेल हो तो बाक़ी फिर भी इंस्टॉल करो
        for line in open('req_poh.txt'):
            line = line.split('#')[0].strip()
            if line and not line.startswith('-'):
                sh(f'pip install -q "{line}"')
    sh('pip install -q -U "protobuf>=5.28" deep-translator')
    open('.poh2_ok', 'w').write('ok')
print('✅ मशीन तैयार')

# ---- 4) WanGP इंजन चालू (मॉडल पहली बार वीडियो बनाते समय डाउनलोड होगा) ----
sys.path.insert(0, WG)
from shared.api import init as wangp_init
PROFILE = '4' if RAM_GB >= 40 else '5'   # कम RAM हो तो "फ़ेल-सेफ़" मोड
OUT = BASE + '/poh_videos'; os.makedirs(OUT, exist_ok=True)
SESSION = wangp_init(root=WG, output_dir=OUT + '/raw', cli_args=['--attention', 'sdpa', '--profile', PROFILE])
print('✅ AI इंजन चालू (profile', PROFILE + ')')

# ---- 5) मॉडल ----
LIGHTNING_14B = {   # Wan 2.2 14B को 4 स्टेप में चलाने वाला Lightning
    "num_inference_steps": 4, "guidance_scale": 1, "guidance2_scale": 1, "guidance_phases": 2,
    "switch_threshold": 876, "model_switch_phase": 1, "flow_shift": 5, "sample_solver": "euler",
    "activated_loras": [
        "https://huggingface.co/DeepBeepMeep/Wan2.2/resolve/main/loras_accelerators/Wan2.2_I2V_A14B_HIGH_lightx2v_MoE_distill_lora_rank_64_bf16.safetensors",
        "https://huggingface.co/DeepBeepMeep/Wan2.2/resolve/main/loras_accelerators/Wan2.2_I2V_A14B_LOW_4steps_lora_rank64_Seko_V1_forKJ.safetensors"],
    "loras_multipliers": "1;0 0;1"}
MODELS = {
    'fast': {'type': 'ti2v_2_2_fastwan', 'extra': {"num_inference_steps": 6, "guidance_scale": 1, "flow_shift": 3},
             'res': {'480': ('832x480', '480x832'), '720': ('1280x720', '720x1280')}, 'min_per_5s': {'480': 5, '720': 11}},
    'best': {'type': 'i2v_2_2', 'extra': LIGHTNING_14B,
             'res': {'480': ('832x480', '480x832'), '720': ('832x480', '480x832')}, 'min_per_5s': {'480': 14, '720': 14}},
}

# ---- 6) हिंदी → अंग्रेज़ी ----
def to_en(text):
    text = (text or '').strip()
    if not text or not re.search(r'[^\x00-\x7F]', text): return text
    try:
        from deep_translator import GoogleTranslator
        return GoogleTranslator(source='auto', target='en').translate(text) or text
    except Exception as e:
        print('⚠️ अनुवाद नहीं हुआ, जैसा लिखा वैसा भेज रहे हैं:', e); return text

MOVES = {   # तैयार मूवमेंट बटन
    '📷 कैमरा धीरे-धीरे पास जाए': 'camera slowly pushes in',
    '📷 कैमरा धीरे-धीरे पीछे जाए': 'camera slowly pulls back',
    '🔄 कैमरा घूमे': 'camera slowly orbits around the subject',
    '💨 बाल और कपड़े हवा में उड़ें': 'hair and clothes gently blowing in the wind',
    '🙂 चेहरा मुस्कुराए / पलक झपके': 'the person blinks naturally and smiles softly',
    '🚶 आगे चले': 'the subject walks forward',
    '🌧️ बारिश': 'rain falling', '⚡ बिजली चमके': 'lightning flashes in the sky',
    '☁️ बादल चलें': 'clouds moving across the sky', '🔥 आग की लपटें': 'flickering flames',
    '🌊 पानी की लहरें': 'water waves moving', '✨ जादुई चमक': 'magical glowing particles floating',
}
SOUNDS = {
    'अपने आप (वीडियो देखकर)': '',
    '🌧️ बारिश और बादल': 'rain and distant thunder', '🌊 समुद्र': 'ocean waves on the shore',
    '🌳 जंगल और पक्षी': 'forest ambience, birds chirping', '💨 तेज़ हवा': 'strong wind blowing',
    '🔥 आग': 'crackling fire', '🏙️ शहर / ट्रैफ़िक': 'city street traffic ambience',
    '👥 भीड़': 'crowd murmur', '🎵 हल्का म्यूज़िक': 'soft cinematic background music',
}

# ---- 7) "AI Generated" निशान ----
from PIL import Image, ImageDraw, ImageFont
def label_png(path, w):
    h = max(28, w // 40); img = Image.new('RGBA', (w, h + 16), (0, 0, 0, 0)); d = ImageDraw.Draw(img)
    try: f = ImageFont.truetype(glob.glob('/usr/share/fonts/**/DejaVuSans*.ttf', recursive=True)[0], h - 8)
    except Exception: f = ImageFont.load_default()
    t = 'AI Generated | Power Orientor Hub'; tw = d.textlength(t, font=f)
    d.rounded_rectangle((w - tw - 34, 6, w - 10, h + 8), 8, fill=(0, 0, 0, 120))
    d.text((w - tw - 22, 9), t, font=f, fill=(255, 255, 255, 215)); img.save(path)

def duration(p):
    try: return float(out(f'ffprobe -v error -show_entries format=duration -of csv=p=0 "{p}"'))
    except Exception: return 0.0
def has_audio(p): return bool(out(f'ffprobe -v error -select_streams a -show_entries stream=index -of csv=p=0 "{p}"'))

def run_job(job, label):
    """WanGP का काम चलाओ और बीच-बीच में हाल बताते रहो।"""
    last = 0
    for ev in job.events.iter(timeout=1):
        if ev.kind == 'progress' and time.time() - last > 3:
            p = ev.data; last = time.time()
            win = f' • हिस्सा {p.window_index}/{p.window_count}' if p.window_count and p.window_count > 1 else ''
            step = f' • स्टेप {p.current_step}/{p.total_steps}' if p.total_steps else ''
            yield f'⏳ {label}{win}{step} • {p.status or p.phase}'
    r = job.result()
    if not r.success or not r.generated_files:
        raise RuntimeError('; '.join(e.message for e in (r.errors or [])) or 'कोई फ़ाइल नहीं बनी')
    yield r.generated_files[-1]

NEG = 'text, watermark, subtitles, extra fingers, deformed face, blurry, bright pink colors, oversaturated, static, frozen'

def banao(photo, prompt_hi, moves, model, quality, seconds, sound, sound_hi, music, agree):
    import gradio as gr
    if not agree: raise gr.Error('पहले नियम वाले बॉक्स पर ✓ लगाओ।')
    if photo is None: raise gr.Error('पहले फ़ोटो डालो।')
    if model == 'best' and not BIG_OK:
        raise gr.Error(f'💎 14B मॉडल के लिए 24 GB RAM और 45 GB डिस्क चाहिए। यहाँ RAM {RAM_GB:.0f} GB, डिस्क {DISK_GB:.0f} GB है। Kaggle में चलाओ या ⚡ तेज़ मॉडल चुनो।')
    if sound == 'music' and not music: raise gr.Error('अपना म्यूज़िक चुना है, तो ऊपर गाना/ऑडियो फ़ाइल डालो।')
    t0 = time.time(); M = MODELS[model]; seconds = int(seconds)
    jid = uuid.uuid4().hex[:8]; jd = f'{OUT}/job_{jid}'; os.makedirs(jd)
    img = Image.open(photo).convert('RGB'); img.save(f'{jd}/in.png'); land = img.width >= img.height

    # प्रॉम्प्ट: हर लाइन = वीडियो का एक हिस्सा (लंबे वीडियो में)
    yield None, '🔤 हिंदी को अंग्रेज़ी में बदल रहे हैं...'
    lines = [l for l in (prompt_hi or '').splitlines() if l.strip()]
    extra = ', '.join(MOVES[m] for m in (moves or []))
    lines = [to_en(l) for l in lines] or ['subtle natural cinematic movement']
    lines = [f'{l}, {extra}' if extra else l for l in lines]
    lines = [l + ', cinematic, natural colors, smooth motion' for l in lines]
    multi = seconds > 5 and len(lines) > 1
    shown = '\n'.join(f'• {l}' for l in lines)

    est = M['min_per_5s'][quality] * -(-seconds // 5)
    yield None, f'⏳ शुरू हुआ। अंदाज़न {est}+ मिनट (पहली बार मॉडल डाउनलोड का समय अलग)।\nAI को भेजा गया प्रॉम्प्ट:\n{shown}'

    st = SESSION.get_default_settings(M['type'])
    st.update(M['extra'])
    st.update({"model_type": M['type'], "prompt": '\n'.join(lines), "negative_prompt": NEG,
               "multi_prompts_gen_type": 'W' if multi else 'FG',
               "resolution": M['res'][quality][0 if land else 1], "video_length": f'{seconds}s',
               "seed": int(time.time()) % 100000, "repeat_generation": 1,
               "image_prompt_type": 'S', "image_start": f'{jd}/in.png', "image_end": None,
               "postprocess_audio": ''})
    json.dump(st, open(f'{jd}/settings.json', 'w'), default=str)

    raw = None
    try:
        for x in run_job(SESSION.submit_task(st), 'वीडियो बन रहा है'):
            if x.startswith('⏳'): yield None, x
            else: raw = x
    except Exception as e:
        raise gr.Error(f'वीडियो नहीं बना: {str(e)[-300:]} — दोबारा कोशिश करो, या छोटा/कम क्वालिटी वाला चुनो।')

    # 🔊 आवाज़
    note = ''
    if sound == 'ai':
        yield None, '🔊 वीडियो देखकर AI आवाज़ बना रहा है (MMAudio, 1-3 मिनट)...'
        sp = SOUNDS[sound_hi] if sound_hi in SOUNDS else to_en(sound_hi)
        try:
            for x in run_job(SESSION.submit_audio_remux(raw, postprocess_audio='mmaudio', postprocess_audio_prompt=sp,
                                                        postprocess_audio_neg_prompt='speech, talking, music with vocals'), 'आवाज़ बन रही है'):
                if x.startswith('⏳'): yield None, x
                else: raw = x
        except Exception as e:
            note = f'\n⚠️ आवाज़ नहीं बन पाई ({str(e)[-120:]}), इसलिए वीडियो बिना आवाज़ के है।'

    # 🎬 फ़ाइनल: 1080p + "AI Generated" + आवाज़/म्यूज़िक
    yield None, '🎬 1080p में फ़ाइनल वीडियो तैयार हो रहा है...'
    W_, H_ = (1920, 1080) if land else (1080, 1920)
    label_png(f'{jd}/lab.png', W_)
    final = f'{OUT}/POH_{jid}.mp4'
    vf = f'[0:v]scale={W_}:{H_}:flags=lanczos,unsharp=5:5:0.7:3:3:0.3[b];[b][1:v]overlay=0:H-h-10[v]'
    enc = '-c:v libx264 -preset veryfast -crf 18 -pix_fmt yuv420p -movflags +faststart'
    if sound == 'music':
        d = duration(raw); fo = max(0.0, d - 2)
        cmd = (f'ffmpeg -v error -y -i "{raw}" -i {jd}/lab.png -stream_loop -1 -i "{music}" -filter_complex '
               f'"{vf};[2:a]atrim=0:{d:.2f},afade=t=in:d=0.5,afade=t=out:st={fo:.2f}:d=2[a]" -map "[v]" -map "[a]" {enc} -c:a aac -b:a 192k -shortest {final}')
    elif has_audio(raw):
        cmd = f'ffmpeg -v error -y -i "{raw}" -i {jd}/lab.png -filter_complex "{vf}" -map "[v]" -map 0:a {enc} -c:a aac -b:a 192k {final}'
    else:
        cmd = f'ffmpeg -v error -y -i "{raw}" -i {jd}/lab.png -filter_complex "{vf}" -map "[v]" {enc} -an {final}'
    if sh(cmd) != 0 or not os.path.exists(final):
        raise gr.Error('फ़ाइनल वीडियो जोड़ने में दिक़्क़त आई। दोबारा कोशिश करो।')
    yield final, f'✅ तैयार! {(time.time()-t0)/60:.1f} मिनट लगे। नीचे वीडियो पर ⋮ या ⬇ दबाकर डाउनलोड करो।{note}'

# ---- 8) ब्रांड वाली स्क्रीन ----
import gradio as gr
CSS = '.gradio-container{max-width:780px!important;margin:auto} #hd{text-align:center} #go{background:#e8a317!important;color:#111!important;font-weight:700;font-size:18px}'
best_label = '💎 बेहतर – Wan 2.2 14B (ज़्यादा अच्छी मूवमेंट, धीमा, 480p)' + ('' if BIG_OK else ' – इस मशीन पर नहीं चलेगा')
with gr.Blocks(title='Power Orientor Hub 2.0', css=CSS, theme=gr.themes.Soft()) as app:
    gr.Markdown('# 🎬 पावर ओरिएंटर हब 2.0\n### फ़ोटो से हिलता वीडियो – अब लंबा, बेहतर और आवाज़ के साथ', elem_id='hd')
    photo = gr.Image(type='filepath', label='1️⃣ फ़ोटो डालो (कार्टून / कॉमिक / सीन / प्रोडक्ट)')
    prompt = gr.Textbox(lines=3, label='2️⃣ क्या हिलना चाहिए? (हिंदी या अंग्रेज़ी – दोनों चलेंगे)',
                        placeholder='जैसे: आदमी धीरे से सिर घुमाता है, आसमान में बिजली चमकती है\n\nलंबे वीडियो में हर लाइन = अगला हिस्सा (हर ~5 सेकंड)')
    moves = gr.CheckboxGroup(list(MOVES), label='या तैयार मूवमेंट चुनो (एक से ज़्यादा चल सकते हैं)')
    with gr.Row():
        model = gr.Radio([('⚡ तेज़ – Wan 2.2 5B', 'fast'), (best_label, 'best')], value='fast', label='3️⃣ मॉडल')
    with gr.Row():
        quality = gr.Radio([('480p – तेज़', '480'), ('720p – साफ़ (सिर्फ़ ⚡ 5B में)', '720')], value='480', label='4️⃣ क्वालिटी')
        seconds = gr.Radio([('5 सेकंड', 5), ('10 सेकंड', 10), ('15 सेकंड', 15), ('20 सेकंड', 20)], value=5, label='5️⃣ लंबाई')
    sound = gr.Radio([('🔇 बिना आवाज़', 'none'), ('🔊 AI आवाज़ (वीडियो देखकर)', 'ai'), ('🎵 अपना म्यूज़िक', 'music')],
                     value='none', label='6️⃣ आवाज़')
    sound_hi = gr.Dropdown(list(SOUNDS), value=list(SOUNDS)[0], allow_custom_value=True, visible=False,
                           label='कैसी आवाज़? (चुनो, या हिंदी में ख़ुद लिखो – जैसे "घोड़े दौड़ रहे हैं")')
    music = gr.Audio(type='filepath', label='अपना गाना / म्यूज़िक डालो (mp3, wav) – सिर्फ़ जिसका हक़ आपके पास हो', visible=False)
    sound.change(lambda s: (gr.update(visible=s == 'ai'), gr.update(visible=s == 'music')), sound, [sound_hi, music])
    agree = gr.Checkbox(label='मैं मानता हूँ: असली लोगों का नक़ली वीडियो, बदनामी, अश्लील या हिंसक कंटेंट नहीं बनाऊँगा।')
    btn = gr.Button('🎬 वीडियो बनाओ', elem_id='go')
    msg = gr.Markdown('')
    vid = gr.Video(label='आपका वीडियो (1080p)')
    btn.click(banao, [photo, prompt, moves, model, quality, seconds, sound, sound_hi, music, agree], [vid, msg])
    gr.Markdown('---\n**अंदाज़न समय (T4 फ़्री GPU, हर 5 सेकंड पर):** ⚡ 5B 480p ≈ 4-6 मिनट • 720p ≈ 9-12 मिनट • 💎 14B 480p ≈ 12-18 मिनट। '
                'पहली बार मॉडल डाउनलोड: 5B ≈ 5 मिनट, 14B ≈ 15-25 मिनट।\n\n'
                '*हर वीडियो पर "AI Generated" लिखा आता है। Claude AI की मदद से बनाया गया। मॉडल: Wan 2.2 (Apache 2.0), MMAudio। इंजन: WanGP (deepbeepmeep)।*')
print('\n🚀 नीचे https://....gradio.live वाला लिंक खोलो। ये सेल चलता रहने दो।\n')
app.queue().launch(share=True, debug=True)
